# Lab 02 – Transactions, Blocks, and Integrity Validation
**Course:** Introduction to Blockchain  |  **Platform:** Google Colab (Linux/Ubuntu runtime)

Run the cells **top to bottom** (Runtime → Run all). Each section has the *logic* explained before the code.

## Step 0 – Check the Linux environment (the "Linux setup" marks)
Colab runs on Ubuntu, so the lab's Linux commands work with a `!` prefix.

In [ ]:
!lsb_release -a
!python3 --version
!pwd

## Step 1 – Create `solution.py`
`%%writefile` saves the cell as a file in the Colab working directory (same as saving `solution.py` on Linux).

**Logic of the program**
1. `digest()` → turns any object into canonical JSON (sorted keys, no spaces) then SHA-256. Canonical JSON guarantees the same data always gives the same hash.
2. `make_block()` → builds `{height, previous_hash, transactions}` and then adds `hash = digest(body)`.
3. `validate()` → for every block: (a) recompute hash of body and compare with stored `hash` (detects edited content); (b) compare `previous_hash` with the previous block's `hash` (detects broken links).
4. Transactions are *admitted* only if the sender balance ≥ amount; otherwise `REJECT insufficient funds`.

In [ ]:
%%writefile solution.py
#!/usr/bin/env python3
"""Week 02: create a toy block chain with account balances and tamper check."""
import argparse, hashlib, json


def digest(obj):
    """Deterministic SHA-256 of any JSON-serialisable object."""
    return hashlib.sha256(
        json.dumps(obj, sort_keys=True, separators=(',', ':')).encode()
    ).hexdigest()


def make_block(index, previous, transactions):
    block = {'height': index, 'previous_hash': previous, 'transactions': transactions}
    block['hash'] = digest(block)          # hash commits to the body
    return block


def validate(chain):
    for i, block in enumerate(chain):
        body = {k: v for k, v in block.items() if k != 'hash'}
        if block['hash'] != digest(body):
            return False, f'block {i} content changed'
        if i and block['previous_hash'] != chain[i - 1]['hash']:
            return False, f'block {i} points to wrong previous block'
    return True, 'hash links correct'


p = argparse.ArgumentParser()
p.add_argument('--tamper', action='store_true')
args = p.parse_args()

balances = {'Alice': 100, 'Bob': 10}
txs = [{'from': 'Alice', 'to': 'Bob', 'amount': 60, 'nonce': 1},   # extension: 60 (was 25)
       {'from': 'Bob', 'to': 'Alice', 'amount': 50, 'nonce': 1}]

accepted = []
for t in txs:
    if balances[t['from']] < t['amount']:
        print('REJECT insufficient funds:', t)
    else:
        balances[t['from']] -= t['amount']
        balances[t['to']] += t['amount']
        accepted.append(t)

chain = [make_block(0, '0' * 64, [{'genesis': True}])]
chain.append(make_block(1, chain[0]['hash'], accepted))
print('ACCEPTED:', accepted, 'BALANCES:', balances)
assert len(accepted) == 2 and balances == {'Alice': 90, 'Bob': 20}
print('CHAIN:', [(b['height'], b['hash'][:12]) for b in chain])
print('VALID BEFORE TAMPER:', validate(chain))
if args.tamper:
    chain[1]['transactions'][0]['amount'] = 999
    print('VALID AFTER TAMPER:', validate(chain))
    assert not validate(chain)[0]
else:
    assert validate(chain)[0]
print('PASS: transaction admission and tamper evidence demonstrated.')

## Step 2 – Run normally (`python3 solution.py`)
**Expected:** both transfers ACCEPTED, balances `{'Alice': 90, 'Bob': 20}`, validation `True`.

In [ ]:
!python3 solution.py

## Step 3 – Run the tamper demonstration (`--tamper`)
After the block hash is recorded, the code changes amount `60 → 999`. `validate()` recomputes SHA-256 and finds a mismatch.
**Expected:** `VALID AFTER TAMPER: (False, 'block 1 content changed')`.

In [ ]:
!python3 solution.py --tamper

## Step 4 – Syntax check (`py_compile`)
No output = success. A syntax check does **not** prove security correctness.

In [ ]:
!python3 -m py_compile solution.py && echo "py_compile OK (exit code 0)"

## Step 5 – Guided Step 1 & 2: predict spending order and admission
| Event | Alice | Bob | Admitted? |
|---|---|---|---|
| Start | 100 | 10 | – |
| Tx1: Alice → Bob 60 | 40 | 70 | Yes (100 ≥ 60) |
| Tx2: Bob → Alice 50 | 90 | 20 | Yes (70 ≥ 50) |

**Comparison (original scenario, 25 units):** after Tx1 Bob has only 35, so Tx2 (50) fails → `REJECT insufficient funds`. Run it below to prove the extension changed the outcome.

In [ ]:
def admit(balances, txs):
    balances = dict(balances); accepted = []
    for t in txs:
        if balances[t['from']] < t['amount']:
            print('REJECT insufficient funds:', t)
        else:
            balances[t['from']] -= t['amount']; balances[t['to']] += t['amount']; accepted.append(t)
    return balances, accepted

start = {'Alice': 100, 'Bob': 10}
mk = lambda a: [{'from':'Alice','to':'Bob','amount':a,'nonce':1},{'from':'Bob','to':'Alice','amount':50,'nonce':1}]

print('--- Original scenario (Alice sends 25) ---')
b, a = admit(start, mk(25)); print('Balances:', b, '| accepted:', len(a))
print('--- Extension (Alice sends 60) ---')
b, a = admit(start, mk(60)); print('Balances:', b, '| accepted:', len(a))

## Step 6 – Guided Step 3 & 5: inspect the block layout and draw the chain
Import the functions and print the real block structure.

In [ ]:
import hashlib, json
def digest(o): return hashlib.sha256(json.dumps(o,sort_keys=True,separators=(',',':')).encode()).hexdigest()
def make_block(i,prev,txs):
    b={'height':i,'previous_hash':prev,'transactions':txs}; b['hash']=digest(b); return b

accepted = [{'from':'Alice','to':'Bob','amount':60,'nonce':1},{'from':'Bob','to':'Alice','amount':50,'nonce':1}]
g  = make_block(0,'0'*64,[{'genesis':True}])
b1 = make_block(1,g['hash'],accepted)
print(json.dumps([g,b1],indent=2))
print()
print(f"GENESIS[{g['hash'][:12]}]  →  BLOCK1[previous_hash={b1['previous_hash'][:12]}; hash={b1['hash'][:12]}]")

## Step 7 – Extra evidence: what *else* breaks the chain?
Two more tests: (a) **reordering** transactions changes `hash1`; (b) fixing block 1's hash after tampering still fails if a **later** block exists (the link breaks).

In [ ]:
# (a) reordering transactions changes the hash
b1_re = make_block(1, g['hash'], list(reversed(accepted)))
print('original hash1 :', b1['hash'][:16])
print('reordered hash1:', b1_re['hash'][:16], '-> different:', b1['hash']!=b1_re['hash'])

# (b) attacker edits block 1 AND recomputes its hash; block 2 exists
def validate(chain):
    for i,blk in enumerate(chain):
        body={k:v for k,v in blk.items() if k!='hash'}
        if blk['hash']!=digest(body): return False,f'block {i} content changed'
        if i and blk['previous_hash']!=chain[i-1]['hash']: return False,f'block {i} points to wrong previous block'
    return True,'hash links correct'

b2 = make_block(2, b1['hash'], [{'from':'Alice','to':'Bob','amount':5,'nonce':2}])
chain = [g, json.loads(json.dumps(b1)), b2]
print('before :', validate(chain))
chain[1]['transactions'][0]['amount'] = 999
chain[1]['hash'] = digest({k:v for k,v in chain[1].items() if k!='hash'})   # attacker re-hashes
print('attacker re-hashed block 1:', validate(chain))

## Step 8 – Viva questions and model answers
**Q1. Submitted vs confirmed?**  Submitted = a candidate transaction not yet in the ledger. Confirmed = admitted into an accepted block under the system rules.

**Q2. Why keep `previous_hash`?**  It links blocks in order. Without it the validator cannot detect a broken predecessor relationship, so blocks could be swapped/removed unnoticed.

**Q3. Does hashing prove the sender approved a transfer?**  No. A hash gives *integrity* only. *Authorization* needs digital signatures. (Anyone can recompute a hash.)

## Common mistakes
* The `nonce` here is just data – the toy code does not enforce nonce order (replay protection).
* Integrity ≠ authorization.
* This is a simplified account model, not a real cryptocurrency protocol (no consensus, signatures, fees or finality).

## Step 9 – Download your evidence
Download `solution.py` and this notebook (File → Download → .ipynb), then take a screenshot of the outputs for your submission.

In [ ]:
from google.colab import files
files.download('solution.py')